In [1]:
import altair as alt
import duckdb

In [2]:
alt.renderers.enable('png')

RendererRegistry.enable('png')

In [3]:
conn = duckdb.connect()

In [4]:
conn.execute("create table if not exists weather_data as select * from 'nyc_weather_data.csv'")
conn.execute("attach 'citibike_data.duckdb' as cb_data;")

In [5]:
conn.sql('describe cb_data.rides')

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ ride_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ rideable_type      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ started_at         │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ ended_at           │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ start_station_name │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ start_station_id   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ end_station_name   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ end_station_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ start_lat          │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │

In [6]:
conn.sql('describe weather_data')

┌──────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name      │ column_type │  null   │   key   │ default │  extra  │
│       varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ date                 │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ temp_max_f           │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ temp_min_f           │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ temp_mean_f          │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ feels_like_max_f     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ feels_like_min_f     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ precipitation_in     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ rain_in              │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ snowfall_in          │ DOUBLE      │ YES     │ NUL

In [7]:
df = conn.execute("""
    from cb_data.rides
    select date_trunc('month',ended_at) as month ,count(1) num_rides
    group by month
""").df()

In [8]:
alt.Chart(df).mark_bar(size=15).encode(
    alt.X('month'),
    alt.Y('num_rides'),
    tooltip=['month', 'num_rides']
).interactive()

ValueError: Saving charts in 'png' format requires the vl-convert-python package: see https://altair-viz.github.io/user_guide/saving_charts.html#png-svg-and-pdf-format

alt.Chart(...)

In [35]:
df = conn.execute("""
    with citibike_grouped_by_day as (
        select date_trunc('day',ended_at) as date,count(1) num_rides
            from cb_data.rides
        group by 1
    )
    select temp_max_f, num_rides, date
    from weather_data join citibike_grouped_by_day using(date)
    order by date asc
""").df()

In [36]:
alt.Chart(df).mark_point().encode(
    x='temp_max_f',
    y='num_rides',
)

alt.Chart(...)